In [4]:
import os
import numpy as np
import torch
import torchvision.transforms as transforms
import torchvision.models as torch_models
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19, ResNet152
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Directories
train_dir = "/kaggle/input/mango-data/Mango_leaf_disease1/train"
valid_dir = "/kaggle/input/mango-data/Mango_leaf_disease1/val"
img_size = (224, 224)
batch_size = 32

def train_tf_model(base_model, model_name):
    """Fine-tune a TensorFlow model (VGG19 / ResNet152)."""
    for layer in base_model.layers:
        layer.trainable = False  # Freeze feature extraction layers

    x = Flatten()(base_model.output)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(num_classes, activation="softmax")(x)
    model = Model(inputs=base_model.input, outputs=x)

    model.compile(loss="categorical_crossentropy", optimizer=Adam(learning_rate=0.0001), metrics=["accuracy"])
    model.fit(train_generator, validation_data=valid_generator, epochs=5, verbose=1)
    
    model.save(f"{model_name}.h5")
    print(f"Model saved: {model_name}.h5")

# Data generators for TensorFlow
train_datagen = ImageDataGenerator(rescale=1.0/255, rotation_range=20, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)
valid_datagen = ImageDataGenerator(rescale=1.0/255)
train_generator = train_datagen.flow_from_directory(train_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical')
valid_generator = valid_datagen.flow_from_directory(valid_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical')
num_classes = len(train_generator.class_indices)

# Train TensorFlow models
train_tf_model(VGG19(weights="imagenet", include_top=False, input_shape=(224, 224, 3)), "VGG19")
train_tf_model(ResNet152(weights="imagenet", include_top=False, input_shape=(224, 224, 3)), "ResNet152")

# PyTorch Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_dataset = ImageFolder(root=train_dir, transform=transform)
valid_dataset = ImageFolder(root=valid_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
num_classes_torch = len(train_dataset.classes)

def train_torch_model(model, model_name):
    """Fine-tune a PyTorch model (AlexNet, GoogleNet, ResNet152)."""
    model = model.to(device)  # Move model to GPU or CPU

    if hasattr(model, 'fc'):
        model.fc = nn.Linear(model.fc.in_features, num_classes_torch).to(device)  # Move to same device
    elif hasattr(model, 'classifier'):
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes_torch).to(device)  # Move to same device

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    model.train()
    
    for epoch in range(5):
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)  # Move data to the same device
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        accuracy = 100 * correct / total
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {accuracy:.2f}%")
    
    # Evaluate on validation set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)  # Move data to the same device
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_accuracy = 100 * correct / total
    print(f"Final validation accuracy for {model_name}: {val_accuracy:.2f}%")
    
    torch.save(model.state_dict(), f"{model_name}.pth")
    print(f"Model saved: {model_name}.pth")


# Load and fine-tune PyTorch models
train_torch_model(torch_models.alexnet(pretrained=True), "AlexNet")
train_torch_model(torch_models.googlenet(pretrained=True), "GoogleNet")
train_torch_model(torch_models.resnet152(pretrained=True), "ResNet152")

Found 2800 images belonging to 8 classes.
Found 800 images belonging to 8 classes.
Epoch 1/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 37s 362ms/step - accuracy: 0.3273 - loss: 1.8703 - val_accuracy: 0.8200 - val_loss: 0.8076
Epoch 2/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 32s 341ms/step - accuracy: 0.6577 - loss: 0.9839 - val_accuracy: 0.8938 - val_loss: 0.4998
Epoch 3/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 33s 341ms/step - accuracy: 0.7606 - loss: 0.7425 - val_accuracy: 0.9162 - val_loss: 0.4079
Epoch 4/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 33s 341ms/step - accuracy: 0.8164 - loss: 0.5877 - val_accuracy: 0.8963 - val_loss: 0.3659
Epoch 5/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 32s 342ms/step - accuracy: 0.8349 - loss: 0.5075 - val_accuracy: 0.9388 - val_loss: 0.2631
Model saved: VGG19.h5
Epoch 1/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 69s 510ms/step - accuracy: 0.1710 - loss: 2.2998 - val_accuracy: 0.3075 - val_loss: 1.8179
Epoch 2/5
88/88 ━━━━━━━━━━━━━━━━━━━━ 34s 361ms/step - accuracy: 0.2253 - loss: 1.9142 - val_accuracy: 0.2925 - val_loss: 1.7927

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=GoogLeNet_Weights.IMAGENET1K_V1`. You can also use `weights=GoogLeNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to /root/.cache/torch/hub/checkpoints/googlenet-1378be20.pth


Model saved: AlexNet.pth


100%|██████████| 49.7M/49.7M [00:00<00:00, 89.2MB/s]


Epoch 1, Loss: 0.6357, Accuracy: 87.71%
Epoch 2, Loss: 0.0621, Accuracy: 99.32%
Epoch 3, Loss: 0.0296, Accuracy: 99.79%
Epoch 4, Loss: 0.0161, Accuracy: 99.86%
Epoch 5, Loss: 0.0103, Accuracy: 99.93%
Final validation accuracy for GoogleNet: 100.00%
Model saved: GoogleNet.pth


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /root/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth
100%|██████████| 230M/230M [00:02<00:00, 87.6MB/s] 


Epoch 1, Loss: 0.2002, Accuracy: 95.89%
Epoch 2, Loss: 0.0310, Accuracy: 99.25%
Epoch 3, Loss: 0.0187, Accuracy: 99.61%
Epoch 4, Loss: 0.0326, Accuracy: 99.11%
Epoch 5, Loss: 0.0174, Accuracy: 99.43%
Final validation accuracy for ResNet152: 99.12%
Model saved: ResNet152.pth
